In [1]:
suppressPackageStartupMessages({
    library(Signac)
    library(Seurat)
    library(GenomicRanges)
    library(data.table)
    library(Matrix)
    library(dplyr)
    library(ggplot2)
    set.seed(123)
    annotations <- readRDS("EnsDb.Hsapiens.v86_annotations_Nov2025.rds")
})

In [2]:
rep1 <- read.table(
  file = "../data/rep1/cellranger/peaks.bed",
  col.names = c("chr", "start", "end")
)
rep2 <- read.table(
  file = "../data/rep2/cellranger/peaks.bed",
  col.names = c("chr", "start", "end")
)

# convert to genomic ranges
gr.rep1 <- makeGRangesFromDataFrame(rep1)
gr.rep2 <- makeGRangesFromDataFrame(rep1)

In [3]:
# Create a unified set of peaks to quantify in each dataset
combined.peaks <- reduce(x = c(gr.rep1, gr.rep2))

# Filter out bad peaks based on length
peakwidths <- width(combined.peaks)
combined.peaks <- combined.peaks[peakwidths  < 10000 & peakwidths > 20]
combined.peaks

GRanges object with 242857 ranges and 0 metadata columns:
             seqnames          ranges strand
                <Rle>       <IRanges>  <Rle>
       [1]       chr1      9897-10800      *
       [2]       chr1     15793-16656      *
       [3]       chr1     17158-17946      *
       [4]       chr1   180960-181794      *
       [5]       chr1   183938-184705      *
       ...        ...             ...    ...
  [242853] KI270728.1 1792061-1792779      *
  [242854] KI270731.1       4514-5416      *
  [242855] KI270734.1   121023-121925      *
  [242856] KI270734.1   131912-132799      *
  [242857] KI270734.1   133504-134383      *
  -------
  seqinfo: 36 sequences from an unspecified genome; no seqlengths

In [4]:
## Create Fragment objects
# load metadata
md.rep1 <- read.table(
  file = "../data/rep1/cellranger/singlecell.csv",
  stringsAsFactors = FALSE, sep = ",", header = TRUE, row.names = 1)[-1, ] 

md.rep2 <- read.table(
  file = "../data/rep2/cellranger/singlecell.csv", 
  stringsAsFactors = FALSE, sep = ",", header = TRUE, row.names = 1)[-1, ] 

# perform an initial filtering of low count cells
md.rep1 <- md.rep1[md.rep1$passed_filters > 500, ]
md.rep2 <- md.rep2[md.rep2$passed_filters > 500, ]

# create fragment objects
frags.rep1 <- CreateFragmentObject(
  path = "../data/rep1/cellranger/fragments.tsv.gz", cells = rownames(md.rep1)
)

frags.rep2 <- CreateFragmentObject(
  path = "../data/rep2/cellranger/fragments.tsv.gz", cells = rownames(md.rep2)
)

Computing hash

Computing hash



In [5]:
## Quantify peaks in each dataset
counts.rep1 <- FeatureMatrix(
  fragments = frags.rep1,
  features = combined.peaks,
  cells = rownames(md.rep1)
)

counts.rep2 <- FeatureMatrix(
  fragments = frags.rep2,
  features = combined.peaks,
  cells = rownames(md.rep2)
)

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions



In [6]:
### Create the objects
assay.rep1 <- CreateChromatinAssay(counts.rep1, fragments = frags.rep1, annotation = annotations)
Gal.rep1 <- CreateSeuratObject(assay.rep1, assay = "ATAC", meta.data=md.rep1)

assay.rep2 <- CreateChromatinAssay(counts.rep2, fragments = frags.rep2, annotation = annotations)
Gal.rep2 <- CreateSeuratObject(assay.rep2, assay = "ATAC", meta.data=md.rep2)


Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from atac to atac_”
Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from atac to atac_”


In [7]:
## Merge objects
# add information to identify dataset of origin
Gal.rep1$duplicate <- 'rep1'; Gal.rep2$duplicate <- 'rep2'

# merge all datasets, adding a cell ID to make sure cell names are unique
Gal <- merge(
  x = Gal.rep1, y = Gal.rep2,
  add.cell.ids = c("rep1", "rep2")
)

In [8]:
Gal <- Gal[,Gal$is__cell_barcode == 1]

In [9]:
saveRDS(Gal, "../output/0_HEK_POLG_galactose_seurat_merged.rds")